# IBRD Portfolio Risk Modeling

This notebook demonstrates the watchlist, anomaly, survival, and forecasting workflow built for the portfolio risk analysis.

In [8]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Project root:', repo_root)

Project root: /home/rigii/ATA/notebooks


In [9]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data_pipeline.silver_layer import clean_ibrd_data

df = clean_ibrd_data()
print(df.shape)
print(df.head(2).to_string())

Cleaned data saved to: /home/rigii/ATA/data/processed/ibrd_clean.csv
(1200, 23)
  Loan Number Country / Economy      Region   Loan Type          Loan Status Board Approval Date  Original Principal Amount (US$)  Cancelled Amount (US$)  Disbursed Amount (US$)  Repaid to IBRD (US$)  Due to IBRD (US$) Agreement Signing Date Effective Date (Most Recent) Last Disbursement Date  loan_age_days  loan_age_years  is_active  is_fully_repaid  is_cancelled  repayment_ratio  disbursed_percent portfolio_status  risk_score
0   LOAN-1000         Indonesia  South Asia  Investment  Disbursing&Repaying          2024-06-18                     4.462225e+08                     0.0            2.659100e+08           33394081.59        56612871.32             2024-01-09                   2025-02-19             2025-12-30            813        2.225873       True            False         False         0.125584          59.591348      Early Stage    0.874416
1   LOAN-1001             India  South Asia   Education 

In [10]:
from src.models.ml_risk import train_default_watchlist_model

result = train_default_watchlist_model(df, threshold=0.7)
print('Metrics:', result['metrics'])
print('Watchlist rows:', len(result['watchlist']))
print(result['watchlist'].head())

Metrics: {'accuracy': 0.84, 'roc_auc': 0.91559510567297, 'average_precision': 0.9085864280228442, 'precision': 0.8407643312101911, 'recall': 0.8516129032258064, 'f1': 0.8461538461538461}
Watchlist rows: 392
  loan_number  predicted_probability risk_band
0   LOAN-2111               0.998536      High
1   LOAN-1270               0.997516      High
2   LOAN-1800               0.997471      High
3   LOAN-1695               0.997140      High
4   LOAN-2033               0.997022      High


In [11]:
from src.models.ml_risk import detect_anomalies

anomalies = detect_anomalies(df, n_outliers=10)
print('Anomalies:', len(anomalies))
print(anomalies[['Loan Number', 'anomaly_score', 'Loan Status']].head())

Anomalies: 10
  Loan Number  anomaly_score Loan Status
0   LOAN-1823       0.669603   Cancelled
1   LOAN-1339       0.661185   Cancelled
2   LOAN-2010       0.640482   Cancelled
3   LOAN-2052       0.639321   Cancelled
4   LOAN-1574       0.635709   Cancelled


In [12]:
from src.models.ml_risk import estimate_repayment_survival

survival = estimate_repayment_survival(df)['survival_summary']
print(survival[['Loan Number', 'loan_age_years', 'repayment_ratio', 'hazard_score', 'expected_repayment_years']].head())

  Loan Number  loan_age_years  repayment_ratio  hazard_score  \
0   LOAN-1000        2.225873         0.125584           1.0   
1   LOAN-1001        2.245038         0.023882           1.0   
2   LOAN-1002       22.105407         0.139932           1.0   
3   LOAN-1003       23.154004         0.413482           1.0   
4   LOAN-1004       22.160164         0.260449           1.0   

   expected_repayment_years  
0                  7.597952  
1                  8.125626  
2                 27.405750  
3                 27.086592  
4                 26.857918  


/home/rigii/ATA/.venv/lib/python3.14/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column Cancelled Amount (US$) have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'Cancelled Amount (US$)'].var())
>>> print(df.loc[~events, 'Cancelled Amount (US$)'].var())

A very low variance means that the column Cancelled Amount (US$) completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/home/rigii/ATA/.venv/lib/python3.14/site-packages/lifelines/utils/__init__.py:1163: ConvergenceWarning: Column loan_age_years has high sample correlation with the duration column. This may harm convergence. This could be a form of 'comp

In [13]:
from src.models.ml_risk import forecast_portfolio_trends

forecast = forecast_portfolio_trends(df, years_ahead=5)
print(forecast)

   Year  Forecasted Outstanding
0  2025            5.881732e+09
1  2026            5.865651e+09
2  2027            5.849569e+09
3  2028            5.833488e+09
4  2029            5.817407e+09


In [14]:
from src.models.ml_risk import build_country_umap

country_projection = build_country_umap(df)
print(country_projection.head())

/home/rigii/ATA/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Country / Economy  total_commitment  total_outstanding  avg_repayment_ratio  \
0        Bangladesh      2.141402e+10       8.831533e+09             0.474006   
1            Brazil      1.659524e+10       7.684850e+09             0.405086   
2          Colombia      1.823079e+10       9.400745e+09             0.483516   
3             Egypt      2.317203e+10       9.374735e+09             0.402772   
4             Ghana      2.052112e+10       8.131863e+09             0.438562   

   avg_risk  loan_count     umap_x    umap_y  
0  0.525994          76  14.908710 -2.999060  
1  0.594914          51  17.519709 -1.168094  
2  0.516484          61  17.060368 -2.558959  
3  0.597228          75  14.238520 -2.300012  
4  0.561438          65  15.180152 -3.757096  


In [16]:
from pathlib import Path

watchlist = result['watchlist']
Path('data/features').mkdir(parents=True, exist_ok=True)
watchlist.to_csv('data/features/loan_watchlist.csv', index=False)
print('Saved watchlist export')

Saved watchlist export


## Summary

This notebook provides a practical base for predictive default risk, anomaly detection, survival-analysis summaries, and portfolio forecasting in the IBRD dataset.